# 03. Layer 3: dự báo lỗi thời

TabPFN v2 (teacher, cần GPU cho nhanh) sinh soft-label out-of-fold, XGBoost (student) học lại. Chọn α trên validation, báo cáo trên test theo thời gian.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "https://github.com/vinhlee9x/conflict-detection-obsolescence-forecasting.git", "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not os.path.exists("src") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Thư mục làm việc:", os.getcwd())

In [ ]:
# Tuỳ chọn: giữ data/ và artifacts/ trên Google Drive giữa các session Colab
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/conflict-detection"
    for folder in ("data/processed", "artifacts"):
        os.makedirs(f"{target}/{folder}", exist_ok=True)
        subprocess.run(f"cp -rn {folder}/. {target}/{folder}/ && rm -rf {folder} && ln -s {target}/{folder} {folder}", shell=True, check=True)

In [ ]:
from src import pipeline, settings
import pandas as pd
settings.TABPFN_MAX_CONTEXT = 3000
summary = pipeline.layer3("kb", teacher="tabpfn")
summary["rows"], summary["positives"], summary["best_alpha"]

In [ ]:
pd.DataFrame(summary["test"])